# `test_integration.py` — Full Pipeline Integration Test

## Purpose
Validates the **complete website-to-model pipeline**: the `TrainedModelAdapter` wraps
`inference.py`'s `SentimentPredictor` and exposes the exact JSON format the frontend expects.

## What is tested
| Check | Detail |
|-------|--------|
| Adapter loads | `TrainedModelAdapter(checkpoint)` succeeds |
| Output format | `predictions` key present; each has `name`, `sentiment`, `confidence` |
| All aspects returned | Every aspect in `aspect_names` appears in each response |
| Sentiment values | Only `negative`, `neutral`, `positive`, `not_mentioned` |
| Confidence range | `0.0 <= confidence <= 1.0` |
| Probabilities sum | `neg + neu + pos ≈ 1.0` (within 3%) |

> **Requires a trained checkpoint** at `ml-research/outputs/cosmetic_sentiment_v1/best_model.pt`.

In [1]:
from tqdm.auto import tqdm
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
import sys, os, time, importlib.util

# ── Path setup ────────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.path.abspath('')
PROJECT_ROOT  = NOTEBOOK_DIR
while not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')):
    parent = os.path.dirname(PROJECT_ROOT)
    if parent == PROJECT_ROOT: break
    PROJECT_ROOT = parent

INFERENCE_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'cosmetic_sentiment_v1', 'evaluation')
SRC_DIR       = os.path.join(PROJECT_ROOT, 'src')

for p in tqdm([PROJECT_ROOT, INFERENCE_DIR, SRC_DIR], desc="Processing p"):
    if p not in sys.path:
        sys.path.insert(0, p)

CKPT = os.path.join(PROJECT_ROOT, 'outputs', 'cosmetic_sentiment_v1', 'best_model.pt')
print('Project root:', PROJECT_ROOT)
print('Checkpoint  :', CKPT)
print('Checkpoint exists:', os.path.exists(CKPT))
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...


Processing p:   0%|          | 0/3 [00:00<?, ?it/s]

Project root: c:\Users\lucif\Desktop\Clearview\ml-research
Checkpoint  : c:\Users\lucif\Desktop\Clearview\ml-research\outputs\cosmetic_sentiment_v1\best_model.pt
Checkpoint exists: True
🕒 Done in 0.01s
✅ Completed: Loading dependencies and libraries.


## Test Cases

10 review texts covering: strongly positive, strongly negative, mixed sentiment,
neutral, short inputs (edge cases), and a detailed long review.

In [2]:
print("⏳ Starting: Defining function check_output...")
import time
_start_time = time.time()
REVIEWS = [
    ('positive_clear',  'This foundation is amazing! Color perfection, stays all day, smells divine.'),
    ('negative_clear',  'Worst lipstick ever. Fades in an hour, chemical stench, packaging snapped.'),
    ('mixed_col_sml',   'Love the color and texture. But the smell is absolutely horrible and fades fast.'),
    ('mixed_pkg_price', 'Great packaging and reasonable price. But skin feels weird and color is patchy.'),
    ('neutral_bland',   'It is an okay product. Nothing special but does what it says.'),
    ('short_great',     'Great!'),
    ('short_terrible',  'Terrible.'),
    ('long_detailed',   (
        'Using this moisturizer for three months. Texture is silky smooth, absorbs well. '
        'Skin feels noticeably softer. Staying power is excellent throughout the day. '
        'However the scent is very strong, almost medicinal — quite unpleasant. '
        'Packaging cracked when dropped. Very expensive for what you get.'
    )),
    ('shipping_late',   'Product arrived two weeks late. Was clearly mis-handled.'),
    ('all_aspects',     (
        'The colour is beautiful, texture is silky, stays all day, smells amazing, '
        'well-priced, delivered quickly and the packaging is premium quality.'
    )),
]

VALID_SENTIMENTS = {'negative', 'neutral', 'positive', 'not_mentioned'}

def check_output(result: dict, name: str, expected_aspects: list) -> list:
    """Validate adapter output format. Returns list of error strings."""
    errors = []
    preds  = result.get('predictions') or result.get('aspects') or []
    if not preds:
        errors.append(f'{name}: Missing predictions/aspects key')
        return errors
    if len(preds) != len(expected_aspects):
        errors.append(f'{name}: Expected {len(expected_aspects)} preds, got {len(preds)}')
    returned = {p.get('name') or p.get('aspect') for p in preds}
    for asp in expected_aspects:
        if asp not in returned:
            errors.append(f'{name}: Missing aspect {asp!r}')
    for p in preds:
        asp_name = p.get('name') or p.get('aspect', '?')
        sent = p.get('sentiment') or p.get('label') or p.get('predicted_class')
        if sent not in VALID_SENTIMENTS:
            errors.append(f'{name}|{asp_name}: Invalid sentiment {sent!r}')
        conf = p.get('confidence', -1)
        if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
            errors.append(f'{name}|{asp_name}: Confidence out of range: {conf}')
        probs = p.get('probs') or p.get('probabilities')
        if probs is not None:
            total = sum(probs.values()) if isinstance(probs, dict) else sum(probs)
            if not (0.97 <= total <= 1.03):
                errors.append(f'{name}|{asp_name}: Probs sum={total:.4f}')
    return errors

print(f'[OK] {len(REVIEWS)} test reviews defined')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function check_output.")


⏳ Starting: Defining function check_output...
[OK] 10 test reviews defined
🕒 Done in 0.00s
✅ Completed: Defining function check_output.


## Run Integration Test

Loads `TrainedModelAdapter` via the `inference_bridge/trained_model_adapter.py` path
and runs all 10 test reviews, checking format on every response.

In [ ]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
if not os.path.exists(CKPT):
    print(f'[SKIP] Checkpoint not found: {CKPT}')
    print('       Train the model first using the Trainer in 09_train.ipynb')
else:
    # ── Load adapter ──────────────────────────────────────────────────────────
    adapter_path = os.path.join(PROJECT_ROOT, 'inference_bridge', 'trained_model_adapter.py')
    spec = importlib.util.spec_from_file_location('trained_model_adapter', adapter_path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    adapter      = mod.TrainedModelAdapter(CKPT)
    aspect_names = adapter.aspect_names
    print(f'[OK] Adapter loaded on: {adapter.device}')
    print(f'     Aspects: {aspect_names}')
    print()

    # ── Run all reviews ───────────────────────────────────────────────────────
    all_errors, successful = [], 0
    print(f'{"Review":<18} {"Time(ms)":>9}  {"Status":<8}  Summary')
    print('-' * 72)

    for name, text in REVIEWS:
        try:
            t0      = time.time()
            result  = adapter.predict(text)
            elapsed = (time.time() - t0) * 1000
            errors  = check_output(result, name, aspect_names)
            all_errors.extend(errors)
            preds   = result.get('predictions') or result.get('aspects') or []
            summary = '  '.join(
                f"{(p.get('name') or p.get('aspect','?'))[:5]}/{(p.get('sentiment','?'))[:3]}"
                for p in preds[:4]
            )
            status = '[OK]' if not errors else '[ERR]'
            print(f'{name:<18} {elapsed:>7.0f}ms  {status:<8}  {summary}')
            if not errors:
                successful += 1
        except Exception as exc:
            print(f'{name:<18}  [PREDICT_ERR] {exc}')
            all_errors.append(f'{name}: {exc}')

    # ── Summary ───────────────────────────────────────────────────────────────
    print()
    print('='*72)
    print('INTEGRATION TEST SUMMARY')
    print('='*72)
    print(f'  Successful : {successful}/{len(REVIEWS)}')
    print(f'  Errors     : {len(all_errors)}')
    if all_errors:
        for e in all_errors:
            print(f'    - {e}')
    print()
    if successful == len(REVIEWS) and not all_errors:
        print(f'[RESULT] PASS — All {len(REVIEWS)} reviews processed correctly.')
    else:
        print(f'[RESULT] FAIL — {len(all_errors)} errors.')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")